In [ ]:
import numpy as np

# copiamios algunas funciones que necesitamos para el ejercicio 13

'''Triangular inferior por filas'''
def sol_trinffil (A, b):
    x = b.copy()
    n = len(b)
    for idx, elem in enumerate(b):
        if elem !=0:
            j = idx
            break
    for i in range(j, n):
        x[i] = (b[i] - A[i, :i] @ x[:i])/A[i,i]
    return x

'''Triangular superior por filas'''
def sol_trsupfil (A, b):
    n = len(b)
    x = b.copy()
    for idx in reversed(range(n)):
        if b[idx] !=0:
            j = idx
            break
    for i in reversed(range(j+1)):
        x[i] = (b[i] - A[i, i+1:] @ x[i+1:])/A[i,i]
    return x

'''Triangular superior por columnas'''
def sol_trsupcol(A, b):
    n = len(b)
    x = b.copy()

    for idx in reversed(range(n)):
        if b[idx] != 0:
            j = idx
            break

    for i in reversed(range(j + 1)):
        x[i] = x[i] / A[i, i]
        x[:i] = x[:i] - A[:i, i] * x[i]

    return x

'''Eliminacion Gaussiana'''
def egaussp(A, b):
    n = A.shape[0]
    U = A.copy()
    y = b.copy()

    for k in range(n):
        if np.max(np.abs(U[k:, k])) != 0:
            # Elegimos el pivot
            l = k + np.argmax(np.abs(U[k:, k]))
            U[[k, l], :] = U[[l, k], :]
            y[[k, l]] = y[[l, k]]
            # Reducir
            v = U[k + 1:, k] / U[k, k]
            U[k + 1:, k] = 0
            U[k + 1:, k + 1:] = U[k + 1:, k + 1:] - np.outer(v, U[k, k + 1:])
            y[k + 1:] = y[k + 1:] - v * y[k]

    return U, y

'''Solucion de un sistema Ax = b usando egaussp'''
def sol_egauss(A, b):
    U, y = egaussp(A, b)
    x = sol_trsupcol(U, y)
    return x

'''Determinante de una matriz'''
def dlup(A):
    n = A.shape[0]
    U = A.copy()
    P = np.eye(n)
    detP = 1
    for k in range(n):
        pivot = k + np.argmax(np.abs(U[k:, k]))
        if pivot != k:
            detP = detP * (-1)
            U[[k, pivot], :] = U[[pivot, k], :]
            P[[k, pivot], :] = P[[pivot, k], :]

        U[k+1: , k] = U[k+1:, k]/U[k, k]
        U[k+1: , k +1:] = U[k+1: , k+1:]-np.outer(U[k+1: , k], U[k, k+1:])

    L = np.tril(U,-1)+np.eye(n)
    U = np.triu(U)

    return U, L, P

'''Inversa de una matriz'''
def inv_lu(A):
    n = A.shape[0]
    I = np.eye(n)
    inv_A = np.zeros((n,n))
    U, L, P = dlup(A)
    for k in range(n):
        y = sol_trinffil(L, P@I[k])
        x = sol_trsupfil(U, y)
        inv_A[:, k] = x
    return inv_A

In [ ]:
# Ejercicio 13

# Matriz mal condicionada
A = np.array([[375. , 374], [752, 750]])
print(f'A={A}')
print('')
# Item a)

# Inversa de A
inv_A = inv_lu(A)
print(f'inv_A={inv_A}')
print('')

# Numero de condicion
norm_A = np.linalg.norm(A, np.inf)
inv_norm_A = np.linalg.norm(inv_A, np.inf)

cond_k = norm_A * inv_norm_A

print(f'cond_k = {cond_k}')
print('')

#ACLARACION: Para perturbar a b denotamos como 'v' a la perturbacion
          #  Para perturbar a sol_x denotamos como 'xi' a la perturbacion

# Item b

# Elegimos un b puede ser random o no
b = np.random.random(2)
#b = np.array([1., 0]) # pueden probar con este b tambien

# Resolvemos el sistema
sol_x = sol_egauss(A, b)

# Usamos los autovectores para perturbar el sistema lineal
autovalores, autovectores = np.linalg.eig(A)

# Usamos el primer autovector asociado a lambda_1
autovector_1 = autovectores[:, 0]

#El item b pide que si perturbamos el b entonces x_sol explota

v = autovector_1 # Perturbamos b con el autovector_1

# error de la perturbacion es ||v|| / ||b||
err_pert1 = np.linalg.norm(v, np.inf)/np.linalg.norm(b, np.inf)

# Podemos obtener que xi = inv_A@v

# error relativo ||xi|| / ||sol_x||
err_rel1 = np.linalg.norm(inv_A@v, np.inf)/np.linalg.norm(sol_x, np.inf)

print('')
print('Perturbamos a b con v')
print('')
print(f'err_perturbacion = {err_pert1}')
print('')
print(f'err_sol_relativo = {err_rel1}')
print('')

# Item c)
# Usamos el segundo autovector de A
autovector_2 = autovectores[:, 1]

#El item c pide que si perturbamos x_sol entonces b explota

xi = autovector_2 # Perturbamos sol_x con el autovector_2

# El error de la perturbacion es ||xi|| / ||sol_x||
err_pert2 = np.linalg.norm(xi, np.inf)/np.linalg.norm(sol_x, np.inf)

# Podemos obtener que v = b - A@(sol_x + xi)

#El error relativo ||v|| / ||b||
err_rel2 = np.linalg.norm(b - A@(sol_x + xi), np.inf)/np.linalg.norm(b, np.inf)
print('')
print('Perturbamos a x_sol con xi')
print('')
print(f'err_sol_perturbacion = {err_pert2}')
print('')
print(f'err_relativo = {err_rel2}')
print('')

print('''IMPORTANTE''')
print('Los autovalores asociados a los vectores son;')
print(f'lamb_1 ={autovalores[0]}')
print('')
print(f'lamb_2 ={autovalores[1]}')
print('')
print('El primer autovalor es pequeño, entonces al perturbar con su autovector asociado podemos ver que no <<explota mucho>> (el error relativo es mayor a 1)')
print('')
print('Por otro lado, el segundo autovalor es muy grande, entonces al perturbar con su autovector asociado podemos ver que si <<explota mucho>> (el error relativo es muy grande)')


A=[[375. 374.]
 [752. 750.]]

inv_A=[[ 375.  -187. ]
 [-376.   187.5]]

cond_k = 846376.9999915324


Perturbamos a b con v

err_perturbacion = 0.8618836471956873

err_sol_relativo = 1.681604617838011


Perturbamos a x_sol con xi

err_sol_perturbacion = 0.0037784636053049716

err_relativo = 1225.5009986900382

IMPORTANTE
Los autovalores asociados a los vectores son;
lamb_1 =0.0017777805871901364

lamb_2 =1124.9982222194128

El primer autovalor es pequeño, entonces al perturbar con su autovector asociado podemos ver que no <<explota mucho>> (el error relativo es mayor a 1)

Por otro lado, el segundo autovalor es muy grande, entonces al perturbar con su autovector asociado podemos ver que si <<explota mucho>> (el error relativo es muy grande)
